# 第 13 集：Pandas 设置值

    > 对应《Numpy & Pandas 数据处理教程》课程。Notebook 按“概念 → 示例 → 观察结果”的顺序整理，建议逐格运行，并尝试修改示例数据。

    ## 本节目标


- 会按标签、位置和条件修改数据
- 理解索引对齐
- 避免链式赋值带来的不确定结果


## 1. 准备数据

为避免影响后续单元格，先创建一张小表。设置值时要明确修改的是原表还是副本。


In [ ]:
import pandas as pd

df = pd.DataFrame(
    {
        "姓名": ["小林", "小周", "小陈", "小吴"],
        "成绩": [88, 92, 59, 76],
        "城市": ["北京", "上海", "北京", "广州"],
    },
    index=["S001", "S002", "S003", "S004"],
)
df


## 2. 修改一个单元格

`at` 按标签，`iat` 按位置。这里只修改现有单元格，没有改变表格结构。


In [ ]:
df.at["S003", "成绩"] = 61
df.iat[3, 1] = 78
df


## 3. 按条件批量修改

推荐在一次 `loc` 操作中同时指定条件和列。下面给低于 80 分的学生加 3 分。


In [ ]:
low_score = df["成绩"] < 80
df.loc[low_score, "成绩"] = df.loc[low_score, "成绩"] + 3
df


## 4. 避免链式赋值

不要写 `df[df["成绩"] < 80]["成绩"] = ...`。第一次选择可能产生临时对象，赋值不一定落回原表。统一写成 `df.loc[条件, 列名] = 新值`。


## 5. 新增列与索引对齐

给一列赋 `Series` 时，Pandas 会按索引标签对齐，而不是只按出现顺序填入。缺少标签的位置会得到缺失值。


In [ ]:
bonus = pd.Series({"S001": 5, "S002": 2, "S004": 4})
df["加分"] = bonus
df


## 6. 用表达式创建列

`assign()` 返回一张新表，适合链式处理；直接写 `df["列名"] = ...` 会修改原表。这里先填补缺失加分，再计算最终成绩。


In [ ]:
result = df.assign(
    加分=df["加分"].fillna(0),
    最终成绩=lambda table: table["成绩"] + table["加分"].fillna(0),
)
result


## 7. 删除行列

`drop(columns=...)` 删除列，`drop(index=...)` 删除行。默认返回新表；需要保留原表时，把结果赋给新变量最清楚。


In [ ]:
without_bonus = result.drop(columns="加分")
without_bonus


## 本节小结

修改数据优先使用一次完整的 `loc`/`at` 操作。赋入 `Series` 时要留意索引对齐，结果出现 `NaN` 往往不是计算错误，而是标签没有匹配。
